# Scalable Urban Mobility Demand Prediction and Hotspot Analysis Using Apache Spark MLlib

## Project Overview
This project implements an end-to-end urban mobility analytics and predictive modeling engine using **Apache Spark (PySpark)**, **Spark SQL**, and **Spark MLlib**.

Using real-world NYC Citi Bike trip transactions (**5,246,236 raw records across 6 monthly dataset files for August 2026**), this system:
1. Ingests and cleans large-scale mobility data via a scalable PySpark ETL pipeline.
2. Validates geographical coordinates, timestamps, and trip durations.
3. Performs temporal, rider-demographic, and bike-type analytics using Spark SQL.
4. Identifies geographic demand hotspots and classifies 2,386 transit stations into High, Medium, and Low demand tiers.
5. Formulates station-level hourly demand incorporating zero-pickup hours.
6. Trains and benchmarks 4 Spark MLlib regression algorithms (**Linear Regression Baseline, Decision Tree, Random Forest, Gradient-Boosted Trees**) using strict chronological train/validation/test splitting.

> **Execution Environment**: Apache Spark 3.5.9 local execution (`local[*]`), Java 8 (build 25.481-b10), Python 3.10.0.

### 1. Initialize PySpark Environment

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is in sys.path whether running from root or notebooks/
for p in [os.path.abspath('.'), os.path.abspath('..')]:
    if p not in sys.path:
        sys.path.insert(0, p)

from src.spark_session import get_spark_session
from src.config import (
    DATASET_DIR, CLEANED_PARQUET_PATH, STATION_HOURLY_PARQUET_PATH,
    STATION_HOTSPOTS_CSV_PATH, REPORTS_DIR, MODELS_DIR
)

spark = get_spark_session("Urban_Mobility_Jupyter_Demo")
print(f"Spark Session Initialized! Spark Version: {spark.version}")

### 2. Scalable Data Ingestion Across All 6 CSV Files

In [ ]:
from src.data_ingestion import load_raw_dataset

df_raw, ingestion_stats = load_raw_dataset(spark, DATASET_DIR)
print(f"Ingested {ingestion_stats['num_files']} files ({ingestion_stats['total_size_mb']} MB)")
print(f"Total raw records: {ingestion_stats['total_records']:,}")
df_raw.printSchema()

### 3. PySpark ETL Pipeline & Data Cleaning
Filters out:
- Trips with duration $\le 1.0$ min (accidental unlocks) or $> 1440$ min (lost bikes).
- Missing start station IDs or names.
- Coordinates outside NYC bounding box ($40.50 \le \text{lat} \le 41.05$, $-74.25 \le \text{lng} \le -73.65$).

In [ ]:
from src.data_cleaning import clean_and_transform_trips, save_cleaned_data

df_clean, cleaning_stats = clean_and_transform_trips(df_raw)
print(f"Raw records:      {cleaning_stats['raw_count']:,}")
print(f"Cleaned records:  {cleaning_stats['cleaned_count']:,}")
print(f"Data retention:   {cleaning_stats['retention_rate_pct']}%")

# Show sample cleaned rows
df_clean.select(
    "ride_id", "started_at", "start_station_name", "trip_duration_min", "pickup_hour", "day_name"
).show(5, truncate=False)

### 4. Advanced SQL Analytics
Evaluating diurnal patterns, weekday vs. weekend dynamics, and rider category differences.

In [ ]:
df_clean.createOrReplaceTempView("trips")

print("=== Weekday vs Weekend Profile ===")
spark.sql("""
    SELECT 
        CASE WHEN is_weekend = 1.0 THEN 'Weekend' ELSE 'Weekday' END AS day_type,
        COUNT(*) AS total_trips,
        COUNT(DISTINCT pickup_date) AS distinct_days,
        ROUND(COUNT(*) / COUNT(DISTINCT pickup_date), 0) AS avg_daily_trips,
        ROUND(AVG(trip_duration_min), 2) AS avg_duration_min
    FROM trips
    GROUP BY is_weekend
""").show(truncate=False)

print("=== Rider Demographics Breakdown ===")
spark.sql("""
    SELECT 
        member_casual,
        COUNT(*) AS total_trips,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_share,
        ROUND(AVG(trip_duration_min), 2) AS avg_duration_min
    FROM trips
    GROUP BY member_casual
""").show(truncate=False)

### 5. Geographic Hotspot Analysis & Quantile Classification

In [ ]:
from src.hotspot_analysis import compute_station_hotspots

station_df, hotspot_summary = compute_station_hotspots(spark, df_clean)

print(f"Total unique stations analyzed: {hotspot_summary['total_unique_stations']:,}")
print(f"Classification boundaries: Low < {hotspot_summary['classification_thresholds']['p30_low_medium_boundary']:.0f} | High >= {hotspot_summary['classification_thresholds']['p80_medium_high_boundary']:.0f}")

print("Top 5 Busiest Stations:")
station_df.select("start_station_name", "total_pickups", "avg_hourly_demand", "demand_classification").show(5, truncate=False)

### 6. Machine Learning Demand Prediction (Spark MLlib)
Features:
- Temporal: `pickup_hour`, `day_of_week`, `day_of_month`, `is_weekend`
- Cyclical: `hour_sin`, `hour_cos`
- Spatial: `latitude`, `longitude`
- Lagged Demand: `lag_1h_demand`, `lag_24h_demand` (strictly historical, partitioned by unique `start_station_id`)

Warm-up & Chronological Splits:
- Warm-up Window: August 01 (24 hours used exclusively for uncorrupted lag initialization; excludes incomplete July 31)
- Train: August 02 - August 23 (79,200 observations, 22 days)
- Validation: August 24 - August 27 (14,400 observations, 4 days)
- Test: August 28 - August 31 (14,400 observations, 4 days)
- Total: 108,000 observations across top 150 stations (30 days × 24 hours)
- Testing Paradigm: Rolling 1-hour-ahead demand forecasting

In [ ]:
from src.model_training import execute_machine_learning_pipeline

ml_results = execute_machine_learning_pipeline(spark, force_recompute_grid=False)
print(f"Selected Best Model: {ml_results['best_model']}")

### 7. Model Performance Comparison Table

In [ ]:
import pandas as pd

comp_data = []
for m_name, vals in ml_results["model_comparison"].items():
    comp_data.append({
        "Model": m_name,
        "Validation RMSE": vals["validation"]["rmse"],
        "Validation MAE": vals["validation"]["mae"],
        "Validation R²": vals["validation"]["r2"]
    })

df_comp = pd.DataFrame(comp_data).sort_values("Validation RMSE")
print("=== Model Benchmark Comparison ===")
print(df_comp.to_string(index=False))

print("\n=== Test Set Holdout Evaluation (Aug 28 - Aug 31) ===")
for m_name, vals in ml_results["test_evaluation"].items():
    print(f"{m_name:28s} -> RMSE: {vals['test_rmse']:.4f} | MAE: {vals['test_mae']:.4f} | R²: {vals['test_r2']:.4f}")

### 8. Gradient-Boosted Trees (GBT) Feature Importance Analysis
Inspect the contribution of each feature to variance reduction across the boosted decision trees.

In [ ]:
import json
from src.config import REPORTS_DIR

fi_path = REPORTS_DIR / "gbt_feature_importance.json"
if fi_path.exists():
    with open(fi_path, "r") as f:
        fi_data = json.load(f)
    print("=== Gradient-Boosted Trees (GBT) Feature Importance Ranking ===")
    df_fi = pd.DataFrame(fi_data)
    df_fi["relative_pct"] = (df_fi["importance"] * 100).round(2)
    print(df_fi.to_string(index=False))
else:
    print("Feature importance report not found. Run model training first.")

### 9. Cleanup and Teardown

In [ ]:
spark.stop()
print("Spark Session stopped successfully.")